[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [70]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
  optimizer.zero_grad()

  for inputs, outputs in micro_batches:

    predictions = model(inputs)
    loss = loss_fn(outputs, predictions) / len(micro_batches)

    loss.backward()
    # print(loss.item())
    total_loss =+ loss.item()

  optimizer.step()

  return total_loss
    # pass  # zero_grad, loop (forward, scale loss, backward), step

In [71]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 0.5957097411155701


In [54]:
# micro = [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)]
# print(micro[2][0].shape)
# print(len(micro))

# for inputs, targets in micro:
#   print(inputs, targets)
#         # # 1. Forward Pass: Get predictions from the data
#         # predictions = model(inputs)

#         # # 2. Calculate Loss: Compare predictions (2, 2) to targets (2, 2)
#         # # We divide by n to average the gradient across all micro-batches
#         # loss = loss_fn(predictions, targets) / n

#         # # 3. Backward Pass: Accumulate gradients
#         # loss.backward()

torch.Size([2, 4])
4
tensor([[-0.2633,  0.1163, -0.6262, -1.3546],
        [ 2.3615,  2.5179, -0.3634,  1.2800]]) tensor([[-0.6828,  1.0573],
        [-1.0479, -2.1567]])
tensor([[ 0.0629,  1.5931,  1.7560, -0.8352],
        [-0.2042,  0.7180,  0.8870, -0.1954]]) tensor([[-0.2956,  0.7723],
        [-1.2598, -0.1854]])
tensor([[ 1.0501,  1.1252,  0.9693,  0.6164],
        [ 0.2336,  1.4415, -0.2153, -0.1148]]) tensor([[-0.4037,  1.2731],
        [-0.2934, -0.7093]])
tensor([[ 1.0394,  0.5540,  0.9226,  0.8940],
        [-0.5573, -0.4264,  0.1181, -0.5596]]) tensor([[ 0.2476, -1.2606],
        [ 0.4292, -0.0676]])


In [72]:
# ✅ SUBMIT
from torch_judge import check, hint
hint('gradient_accumulation')
check('gradient_accumulation')


💡 Hint for Gradient Accumulation:
   Zero grads once. For each micro-batch: forward, loss/n_batches, backward. Then optimizer.step(). The loss scaling ensures accumulated grads match a single large batch.


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (6.3ms)
  ✅ [2/3] Returns loss value (1.2ms)
  ✅ [3/3] Parameters actually update (0.9ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (8.3ms total)
  Progress saved. Run status() to see your dashboard.

